In [ ]:
import json
import os
import random

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from shapely.geometry import Polygon

In [ ]:
ground_truths = {}

with open("data/ocr_test_20251127/test.txt") as fin:
    for line in fin:
        line = line.strip()
        if not line:
            break
        im_name, gt = line.split('\t')
        gt = json.loads(gt)
        ground_truths[os.path.basename(im_name)] = gt

print(len(ground_truths))

In [ ]:
predictions = {}

with open("output/predictions/predict_det.txt") as fin:
    for line in fin:
        line = line.strip()
        if not line:
            break
        im_name, pred = line.split('\t')
        pred = json.loads(pred)
        predictions[os.path.basename(im_name)] = pred

print(len(predictions))

In [ ]:
def polygon_iou(pts_a, pts_b):
    """IoU between two polygons given as lists of [x, y] points."""
    poly_a = Polygon(pts_a)
    poly_b = Polygon(pts_b)
    if not poly_a.is_valid:
        poly_a = poly_a.buffer(0)
    if not poly_b.is_valid:
        poly_b = poly_b.buffer(0)
    inter = poly_a.intersection(poly_b).area
    union = poly_a.union(poly_b).area
    return inter / union if union > 0 else 0.0


IOU_THRESHOLD = 0.3

def match_detections(gt_list, pred_list, iou_threshold=IOU_THRESHOLD):
    """
    Greedy IoU-based matching between GT and predicted polygons.

    Returns:
        matched_gt   - set of GT indices matched (TP)
        matched_pred - set of pred indices matched (TP)
        fn_idx       - list of unmatched GT indices (missed detections)
        fp_idx       - list of unmatched pred indices (redundant detections)
    """
    gt_pts   = [item["points"] for item in gt_list]
    pred_pts = [item["points"] for item in pred_list]

    matched_gt   = set()
    matched_pred = set()

    iou_pairs = []
    for gi, gp in enumerate(gt_pts):
        for pi, pp in enumerate(pred_pts):
            iou = polygon_iou(gp, pp)
            if iou >= iou_threshold:
                iou_pairs.append((iou, gi, pi))

    iou_pairs.sort(reverse=True)
    for iou, gi, pi in iou_pairs:
        if gi not in matched_gt and pi not in matched_pred:
            matched_gt.add(gi)
            matched_pred.add(pi)

    fn_idx = [i for i in range(len(gt_pts))   if i not in matched_gt]
    fp_idx = [i for i in range(len(pred_pts)) if i not in matched_pred]

    return matched_gt, matched_pred, fn_idx, fp_idx

In [ ]:
# Compute per-image TP / FP / FN and accumulate global counts
total_tp = total_fp = total_fn = 0
per_image_results = {}

for im_name, gt_list in ground_truths.items():
    pred_list = predictions.get(im_name, [])
    matched_gt, matched_pred, fn_idx, fp_idx = match_detections(gt_list, pred_list)

    tp = len(matched_gt)
    fp = len(fp_idx)
    fn = len(fn_idx)
    total_tp += tp
    total_fp += fp
    total_fn += fn

    per_image_results[im_name] = {
        "gt":           gt_list,
        "pred":         pred_list,
        "matched_gt":   matched_gt,
        "matched_pred": matched_pred,
        "fn_idx":       fn_idx,
        "fp_idx":       fp_idx,
    }

precision = total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0.0
recall    = total_tp / (total_tp + total_fn) if (total_tp + total_fn) > 0 else 0.0
f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

n_images      = len(per_image_results)
n_missed      = sum(1 for r in per_image_results.values() if r["fn_idx"])
n_redundant   = sum(1 for r in per_image_results.values() if r["fp_idx"])

print(f"IoU threshold  : {IOU_THRESHOLD}")
print(f"Images         : {n_images}")
print()
print(f"TP={total_tp}  FP={total_fp}  FN={total_fn}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1        : {f1:.4f}")
print()
print(f"Images with ≥1 missed detection  (FN): {n_missed:4d}  ({100*n_missed/n_images:.1f}%)")
print(f"Images with ≥1 redundant detection (FP): {n_redundant:4d}  ({100*n_redundant/n_images:.1f}%)")

In [ ]:
IMAGE_DIR = "data/ocr_test_20251127/images"

GRID_COLS = 2
GRID_ROWS = 4

def show_samples(case_type="missed", n=GRID_COLS * GRID_ROWS, do_random: bool = False):
    """
    Display sample images highlighting detection errors in a 2-col x 4-row grid.

    Parameters
    ----------
    case_type : "missed" | "redundant"
        "missed"    -> images with FN (GT boxes the model failed to detect)
        "redundant" -> images with FP (predicted boxes with no matching GT)
    n         : number of images to show (defaults to fill the grid)
    do_random : if True, randomly sample from candidates; otherwise take the first n

    Colour legend
    -------------
    lime   - GT box matched by a prediction  (TP)
    red    - GT box with no matching pred    (FN / missed)
    cyan   - predicted box matched to a GT   (TP)
    orange - predicted box with no GT match  (FP / redundant)
    """
    from matplotlib.lines import Line2D

    assert case_type in ("missed", "redundant"), "case_type must be 'missed' or 'redundant'"
    key = "fn_idx" if case_type == "missed" else "fp_idx"
    candidates = [(name, res) for name, res in per_image_results.items() if res[key]]

    if do_random:
        samples = random.sample(candidates, min(n, len(candidates)))
    else:
        samples = candidates[:n]

    if not samples:
        print(f"No images with {case_type} detections found.")
        return

    print("Filenames")
    for name, _ in samples:
        print(f"\t{name}")
    
    n_cols = GRID_COLS
    n_rows = (len(samples) + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(6 * n_cols, 5 * n_rows))
    axes = np.array(axes).reshape(-1)  # flatten for easy indexing

    for i, ax in enumerate(axes):
        if i >= len(samples):
            ax.axis("off")
            continue

        im_name, res = samples[i]
        img_path = os.path.join(IMAGE_DIR, im_name)
        try:
            img = Image.open(img_path)
        except FileNotFoundError:
            ax.set_title(f"Not found\n{im_name[:30]}", fontsize=7)
            ax.axis("off")
            continue

        ax.imshow(img)

        for gi, item in enumerate(res["gt"]):
            pts = np.array(item["points"])
            color = "lime" if gi in res["matched_gt"] else "red"
            ax.add_patch(plt.Polygon(pts, fill=False, edgecolor=color, linewidth=1.5))

        for pi, item in enumerate(res["pred"]):
            pts = np.array(item["points"])
            color = "cyan" if pi in res["matched_pred"] else "orange"
            ax.add_patch(plt.Polygon(pts, fill=False, edgecolor=color, linewidth=1.5))

        ax.set_title(
            f"{im_name[:35]}\nFN={len(res['fn_idx'])}  FP={len(res['fp_idx'])}",
            fontsize=7,
        )
        ax.axis("off")

    legend_elements = [
        Line2D([0], [0], color="lime",   lw=2, label="GT matched (TP)"),
        Line2D([0], [0], color="red",    lw=2, label="GT missed (FN)"),
        Line2D([0], [0], color="cyan",   lw=2, label="Pred matched (TP)"),
        Line2D([0], [0], color="orange", lw=2, label="Pred redundant (FP)"),
    ]
    title = "Missed detections (FN)" if case_type == "missed" else "Redundant detections (FP)"
    fig.legend(handles=legend_elements, loc="lower center", ncol=4, fontsize=9)
    plt.suptitle(title, fontsize=12)
    plt.tight_layout(rect=[0, 0.03, 1, 1])
    plt.show()

In [ ]:
# Missed detections: GT boxes the model failed to find (shown in red)
show_samples(case_type="missed", n=8)

In [ ]:
# Redundant detections: predicted boxes with no matching GT (shown in orange)
show_samples(case_type="redundant", n=8)